In [1]:
from __future__ import annotations

import argparse
import gc
import logging
import os
import sys

import numpy as np
import torch
from drcnet_hybrid_2.data import TrainingDataSet
from drcnet_hybrid_2.fit import fit_model
from drcnet_hybrid_2.model import DenoiserNet
from drcnet_hybrid_2.reconstruction import (
    reconstruct_dwis,
    reconstruct_full_dwi_chunked,
    reconstruct_full_dwi_static_base,
    reconstruct_dwis_index_volume,
)
from drcnet_hybrid.run import fit_progressive
from torch.utils.data import DataLoader, Subset
from utils import setup_logging
from utils.checkpoint import load_checkpoint
from utils.data import (
    StanfordDataLoader,
    compute_brain_mask,
    rescale_reconstruction_to_01,
)
from utils.metrics import (
    compute_metrics,
    fully_compare_volumes,
    save_metrics,
    visualize_single_volume,
)
from utils.multi_gpu import create_multi_gpu_config_from_dict, setup_multi_gpu
from utils.utils import load_config, noise_path_segment

import wandb


def _build_checkpoint_dir(settings, train_num_volumes: int) -> str:
    noise_segment = noise_path_segment(
        getattr(settings.data, "noise_type", "rician"),
        getattr(settings.data, "noise_sigma", 0.1),
    )
    bvalue_segment = f"b{getattr(settings.data, 'bvalue', 2500)}"
    return os.path.join(
        settings.train.checkpoint_dir,
        bvalue_segment,
        f"num_volumes_{train_num_volumes}",
        noise_segment,
        f"learning_rate_{settings.train.learning_rate}",
    )


def _prepare_stanford_arrays(settings, train_num_volumes: int):
    d = settings.data
    data_loader = StanfordDataLoader(
        bvalue=d.bvalue,
        noise_sigma=d.noise_sigma,
    )
    original_data, noisy_data = data_loader.load_data()
    if original_data is None:
        original_data = noisy_data
        logging.info(
            "StanfordDataLoader returned original_data=None; using noisy_data as reference"
        )

    tx, ty, tz = d.take_x, d.take_y, d.take_z
    cropped_noisy = noisy_data[:tx, :ty, :tz, :].copy()
    cropped_orig = original_data[:tx, :ty, :tz, :].copy()

    n_b0 = int(d.num_b0s)
    dwi_noisy = cropped_noisy[..., n_b0:]
    dwi_orig = cropped_orig[..., n_b0:]
    n_dwi = dwi_noisy.shape[-1]
    logging.info(
        f"After spatial crop and b0 skip: DWI shape (X,Y,Z,V)={dwi_noisy.shape}, "
        f"V={n_dwi} diffusion volumes (excluding {n_b0} b0s)"
    )

    if train_num_volumes > n_dwi:
        raise ValueError(
            f"train_num_volumes={train_num_volumes} exceeds available DWI volumes ({n_dwi})"
        )

    train_noisy = dwi_noisy[..., :train_num_volumes]
    train_orig = dwi_orig[..., :train_num_volumes]

    return train_noisy, train_orig, dwi_noisy, dwi_orig


script_dir = "/teamspace/studios/this_studio/TechJourney/DWMRI/src/drcnet_hybrid_2"
config_path = os.path.join(script_dir, "config.yaml")
settings = load_config(config_path).dbrain

train_num_volumes = int(getattr(settings.data, "train_num_volumes", settings.data.num_volumes))
reconstruct_full = bool(getattr(settings.data, "reconstruct_full_dwi", True))

settings.data.num_volumes = train_num_volumes
settings.model.in_channel = train_num_volumes

d = settings.data
spatial_full = (d.take_x, d.take_y, d.take_z)
rec_dev = settings.reconstruct.device

In [2]:
spatial_full

(128, 128, 96)

In [3]:
from utils.data import DBrainDataLoader

data_loader = DBrainDataLoader(
    nii_path=settings.data.nii_path,
    bvecs_path=settings.data.bvecs_path,
    bvalue=settings.data.bvalue,
    noise_sigma=settings.data.noise_sigma,
    noise_type=getattr(settings.data, "noise_type", "rician"),
    n_coils=getattr(settings.data, "noise_n_coils", 1),
)
original_data, noisy_data = data_loader.load_data()

In [4]:
rec_model = DenoiserNet(
    input_channels=train_num_volumes,
    output_channels=settings.model.out_channel,
    groups=settings.model.groups,
    dense_convs=settings.model.dense_convs,
    residual=settings.model.residual,
    base_filters=settings.model.base_filters,
    output_shape=(
        settings.model.out_channel,
        spatial_full[0],
        spatial_full[1],
        spatial_full[2],
    ),
    device=settings.train.device,
    output_activation=getattr(settings.model, "output_activation", "prelu"),
)

In [ ]:
checkpoint_dir = "/teamspace/studios/this_studio/drcnet_hybrid_2/checkpoints/dbrain/b2500/num_volumes_10/noise_rician_sigma_0.1/learning_rate_0.0003/"
best_ckpt = os.path.join(checkpoint_dir, "best_loss_checkpoint.pth")

rec_model, _, _, _, _, _ = load_checkpoint(
    model=rec_model,
    optimizer=None,
    filename=best_ckpt,
    device=rec_dev,
    strict=False,
)

/teamspace/studios/this_studio/TechJourney/DWMRI/src/utils/checkpoint.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=tor

In [6]:
take_volumes = settings.data.num_b0s + settings.data.num_volumes
noisy_data = noisy_data[
    : settings.data.take_x,
    : settings.data.take_y,
    : settings.data.take_z,
    settings.data.num_b0s :,
]
original_data = original_data[
    : settings.data.take_x,
    : settings.data.take_y,
    : settings.data.take_z,
    settings.data.num_b0s :,
]

In [7]:
noisy_data.shape

(128, 128, 96, 60)

In [8]:
reconstructed = reconstruct_full_dwi_chunked(
    model=rec_model,
    noisy_xyzv=noisy_data[..., :15],
    train_num_volumes=train_num_volumes,
    device=rec_dev,
    mask_p=settings.reconstruct.mask_p,
    n_preds=settings.reconstruct.n_preds,
)

In [9]:
ref_for_metrics = original_data
noisy_for_viz = noisy_data
reconstructed = reconstructed

In [10]:
# shape = (2, 3, 0, 1)

# fully_compare_volumes(
#     original_volume=np.transpose(ref_for_metrics, shape),
#     noisy_volume=np.transpose(noisy_for_viz, shape),
#     denoised_volume=np.transpose(reconstructed, shape),
#     file_name=None,
#     volume_idx=20,
# )

In [11]:
reconstructed = reconstruct_full_dwi_static_base(
    model=rec_model,
    noisy_xyzv=noisy_data[..., :15],
    train_num_volumes=train_num_volumes,
    device=rec_dev,
    mask_p=settings.reconstruct.mask_p,
    n_preds=settings.reconstruct.n_preds,
)

RuntimeError: Given groups=1, weight of size [32, 10, 3, 3, 3], expected input[1, 5, 128, 128, 96] to have 10 channels, but got 5 channels instead